# Pipeline v3

Iteration 3 of the lecture pipeline, built from scratch. This notebook documents every stage and script, shows how to
run each one, and records the decisions and runs made along the way.

Run it from the `pipeline/` folder with the repo's Python (no packages beyond the standard library are needed).

| Folder | What is in it |
|---|---|
| `input/<course>/input/` | `subs/` (SRT per lecture), `transcripts/`, `titles.tsv`, `course.json` (course metadata, from the syllabus) |
| `input/<course>/media/` | lecture videos `L01.mp4` … |
| `prompts/` | prompt texts (`segmentation.txt`, `fact_extraction.txt`, `fact_judge_*.txt`, `fact_rewrite.txt`, …) |
| `configs/` | saved `LLMConfig` files |
| `scripts/` | `s0a_preprocess.py`, `s1a_segmentation.py`, `s1b_verify_segments.py`, `s1c_segment_texts.py`, `s2a_fact_extraction.py`, `s2b_verify_facts.py`; helpers `llm/`, `course_meta.py`, `lectures.py`, `fact_checks.py`, `view.py` |
| `output/<course>/` | one folder per step: `stage_0a_preprocess/`, `stage_1a_segmentation/`, `stage_1b_verify_segments/`, `stage_1c_segment_texts/`, `stage_2a_fact_extraction/`, `stage_2b_verify_facts/` |
| `output/logs/` | run logs (`segmentation_run<k>.log`, `verify_run<k>.log`) and model server logs (`llm_server_<port>.log`) |

Stages so far. Phase 1 (0a–1c) segments the lectures; phase 2 builds knowledge: 2a fact extraction, 2b fact
verification, 2c concept extraction (not built yet).

| Stage | Script | Input | Output |
|---|---|---|---|
| 0a Preprocess | `s0a_preprocess.py` | SRT files | sentences with IDs, and their times |
| 1a Segmentation | `s1a_segmentation.py` | 0a sentences | the model's segments (start sentence + sub-goal), checked by 1b's checks with retries |
| 1b Verify segments | `s1b_verify_segments.py` | 1a answers | deterministic checks; segment maps (segment → sentence range, time, sub-goal) |
| 1c Segment texts | `s1c_segment_texts.py` | 1b segment maps | one `.txt` per segment |
| 2a Fact extraction | `s2a_fact_extraction.py` | 1b segment maps, 1c segment texts | the facts of each segment (text + evidence sentences); code checks |
| 2b Verify facts | `s2b_verify_facts.py` | 2a facts and 2a's answers | facts judged on eight properties; failed ones rewritten or removed; final fact maps |
| – Browsing | `view.py` | all outputs | dicts for the notebook (`sentences["1/0"]`, `segments["1/S0"]`) |

## Record: decisions so far

**Inputs (2026-11-09).** The VR course (NPTEL *Foundation for Virtual and Augmented Reality Systems*, 63 lectures) is
in `input/nptel_vr/`. A copy stays in `dataset/virtual_and_augmented_reality/`. Every course's subtitles and
transcripts (NPTEL and MIT) are human-written and verified, so their punctuation can be trusted.

- Lectures 4 and 5 have no NPTEL subtitle file. Their `lec_4.srt` / `lec_5.srt` come from pipeline v1: the official
  transcript, one sentence per cue, timed by aligning it to YouTube's automatic captions.
- Videos: `input/nptel_vr/media/L01.mp4` … `L63.mp4`.

**Earlier segmentations not reused.** v1's segments and v2's (large run, Qwen 3.8 27B) were looked at. v2's large run
had 321 parts, 36 of them over 10 min and one of 28 min (L51/P04: a whole worked problem that the "stand alone" rule
would not cut). Run 1 started again from v2's prompt unchanged.

**Segmentation prompt rewritten (2026-11-11, used from run 2).** A segment is defined by its sub-goal:
1. A segment is where the lecturer completes one sub-goal with respect to the whole lecture. Greetings and endings are
   not sub-goals (they are not lecture content): opening remarks go into the first segment, closing remarks into the last.
2. Cut where the sub-goal is achieved and closed, not at a bridge: the bridge goes to the next segment. Each segment must
   stand alone and be watchable independently: no deixis (no pointing back to the previous segment).
3. No length limit. (A 6–15 min rule was considered and dropped: run 1 had 473 of 545 segments under 6 min, so code
   would have done nearly all the segmenting.)

Added: course and lecture titles in the prompt; sub-goals at the level of the lecture's sections; a working method
(goal → sub-goals → cuts → check); the answer gives each segment's start and its sub-goal, so the model commits to what
each segment achieves and we can judge segments at a glance.

**Times of segment boundaries inside a subtitle cue (2026-11-11).** In run 4, 3 cuts (6 segments) fell inside a cue,
leaving those segments without a time. Options considered: estimate within the cue; force-align the audio (exact, but a
heavy stage for a few cuts); snap the cut to the cue's edge (rejected: moves a sentence between segments, i.e. code
changing the segmentation); play from the cue's start (rejected: the learner hears 5–15 s of the previous segment).
Chosen: estimate, for segment boundaries only, from the course's median speaking rate (143 words per minute, from stage
0a). Estimating from one side drifted 4–5 s when the nearest exact time was far (a sentence that began in an earlier
cue: 48–54 s away); from both sides, averaged, the 4 test boundaries came within 0.2–2.7 s of their position in the cue.

| Boundary | Exact times around it | Forward only | Both sides | Position in the cue |
|---|--:|--:|--:|--:|
| 1/31 | 11 s apart | 201.6 | 201.6 | 201.8 |
| 33/11 | 54 s apart | 126.0 | 132.4 | 129.7 |
| 42/240 | 15 s apart | 2120.5 | 2122.8 | 2121.1 |
| 57/206 | 48 s apart | 2127.1 | 2131.0 | 2132.1 |

**Boundary times from the cue's edges (2026-11-11).** The estimate from the nearest exact sentence times drifted in
courses whose cues are fixed-length lines (Programming in C: 90th percentile 14 s, max 133 s). Stage 0a now saves each
lecture's cue timeline and stage 1b estimates from the edges of the cue the boundary is in: median 0.2–0.6 s, 90th
percentile under 1.8 s in all six courses. VR's segments are unchanged; one estimated time moved (57/S4–S5: 2130.96 →
2133.16 s). Reports: `analysis/preprocessing/report.md`, `analysis/segmentation/report.md`.

**Other courses (2026-11-11).** Stage 0a was extended for the MIT courses (lecture keys like `24b` and `tut_1`, speaker
labels, sound tags, the OCW licence notice) and tested on all five other dataset courses, without saving their outputs;
the VR output is byte-identical and its prompts unchanged. Run a new course with the pipeline after writing its
`course.json`. Boundary-time estimates (median error, 90th percentile) against the boundary's place in its cue:
VR 0.9 s / 3.3 s; MIT 18.06 0.6 / 2.2; MIT 9.00SC 0.4 / 1.2; MIT 6.7960 0.6 / 1.9. In the two other NPTEL courses
the cues are fixed-length lines that split sentences anywhere (C: 7,241 of 9,180 sentence starts inside a cue), so the
nearest exact time can be far away: Introduction to ML 3.8 / 24.8 s (max 158 s), Programming in C 3.5 / 14.2 s (max
133 s). The Introduction to ML subtitles also have few full stops (32 words per sentence at the median, up to 476).

**Prompt revised after run 3 (2026-11-11, used from run 4).** Run 3 (12B) followed the stand-alone rule literally and
merged segments over connectives ("So, given this background…"), giving 28-min segments.
- Stand-alone now forbids only *real* dependencies: the segment needs something only the previous segment explains and
  never names or restates ("these three values" not given again, "this algorithm" never named). Connectives and short
  recaps are allowed; never merge or move a cut because of a connective; when there is a real dependency, look for a
  nearby cut first and merge only as a last resort. ("Move it back to where that thought begins" is gone.)
- The closing remarks (summary, preview, goodbye) may be the last segment on their own. Opening remarks still go into
  the first segment.

**Prompt restructured (2026-11-11, from run 4).**
- The task comes first, stated formally: divide the lecture into segments, each the part in which the learner achieves
  one sub-goal, each watchable independently of the others.
- The course block follows: the course's **outline and syllabus only** ("use this to understand what the course covers
  as a whole"). Nothing platform-specific (no institution, instructor, audience, prerequisites, weeks): the prompt must
  work for NPTEL and MIT alike. The syllabus is converted by hand from `dataset/<course>/course_meta/syllabus.pdf` to
  `course.json`, which keeps everything in the PDF as a record; `prompts/course_meta.txt` uses only `outline` and
  `syllabus` (`scripts/course_meta.py`). The syllabus is shown as topics in order, grouped by theme.
- Then only the lecture's title ("This lecture is titled …"); no place in the course, previous or next lecture.
- **Sub-goals are defined by the course goals, not by the video's structure.** A sub-goal is a step toward the course
  goals that the lecture achieves: something the learner can do afterwards that they could not before; in order, the
  learner's task list for the lecture. A sub-goal is a whole step: the examples, illustrations, details and
  digressions used to achieve it are part of it. (Replaces the list rule "items of a list under one heading make one
  sub-goal", which was too specific and seemed to prompt the model to split items.) Opening and closing remarks are
  not sub-goals because they are about the lecture itself, not the course goals.
- The closing remarks may be the last segment on their own, with a sub-goal saying what they do ("Summarize …").

In [ ]:
%load_ext autoreload
%autoreload 2
import json
from pathlib import Path
from IPython.display import Markdown, display

import sys
assert Path("scripts").is_dir(), "run this notebook from the pipeline folder"
sys.path.insert(0, str(Path.cwd()))   # so that "from scripts import ..." works
COURSE = "nptel_vr"
OUT = Path("output") / COURSE

## Stage 0a: preprocess

`scripts/s0a_preprocess.py` turns each lecture's SRT into numbered sentences.

- All cue text of a lecture is joined and split at sentence ends (`.` `?` `!`, plus any closing quote or bracket) that
  are followed by a space and a capital letter, digit or opening quote. A sentence running across cues stays whole; a
  cue holding several sentences is split. `Dr.`, `Prof.`, `Mr.`, `e.g.`, `i.e.`, `vs.`, `Fig.`, `Eq.` do not end a
  sentence. `MR.` does (in this course it is Mixed Reality).
- A lowercase word after a full stop does not start a sentence. In the VR course that full stop is mid-sentence
  ("…technologies. which, …", 286 times); in MIT 18.06 it is a missed capital ("OK. let's …"). Either way, at worst two
  sentences stay one.
- Sentence IDs are `<key>/<i>`, with `i` from 0 within the lecture: `1/0`, `1/1`, … The key is the lecture's name
  from its file (`scripts/lectures.py`): `lec_7.srt` → `7`, `lec_24b.srt` → `24b`, `tutorials/tut_1.srt` → `tut_1`.
  Every stage names a lecture's files after its subtitle file (`lec_24b.json`, `tut_1.json`); lectures sort by number
  (24 before 24b), then tutorials.

**Caption conventions** (MIT OpenCourseWare; nothing changes for a course without them):
- Speaker labels (`PROFESSOR:`, `AUDIENCE:`, `PHILLIP ISOLA:`, `A. STRANG:`): a speaker turn always starts a new
  sentence, with its label in front. A label is ALL-CAPS words with at least 4 letters, then a colon (`VR:` in the VR
  course is not one).
- Sound tags (`[LAUGHTER]`, `[CLICKING]`) are removed; a cue left empty is dropped. `[INAUDIBLE]` stays (it stands for
  words that were said).
- The OCW licence notice that opens every MIT 9.00SC lecture is dropped.

| File | Content |
|---|---|
| `output/<course>/stage_0a_preprocess/<stem>.json` (`lec_7.json`, `tut_1.json`) | `{"1/0": "Hello, welcome to …", …}` |
| `output/<course>/stage_0a_preprocess/times/<stem>.json` | `{"1/0": {"start": 24.297, "end": 31.884}, …}` |
| `output/<course>/stage_0a_preprocess/times/cues/<stem>.json` | the cue timeline, for stage 1b's time estimates: `{"skip_words": 0, "cues": [[start, end, words], …]}` |
| `output/<course>/stage_0a_preprocess/times/meta.json` | speaking rate: `wpm_median` (median over all cues of the course of words per minute; VR: 143.0), `lecture_wpm_median` |

**Times.** A sentence that begins at the start of a cue gets that cue's start time; one that ends at the end of a cue
gets that cue's end time. When a sentence boundary falls inside a cue, that side has no time of its own and is `null`.
A `null` start means the sentence begins inside the cue where the previous sentence ends (whose end is `null` too). In
the VR course 201 sentences start inside a cue (38 of them also end inside one). No times are made up.

Command line: `python scripts/s0a_preprocess.py nptel_vr`

In [ ]:
from scripts import s0a_preprocess
stats = s0a_preprocess.run(COURSE)

In [ ]:
s0 = OUT / "stage_0a_preprocess"
sents = json.loads((s0 / "lec_1.json").read_text())
times = json.loads((s0 / "times" / "lec_1.json").read_text())
for k in list(sents)[:3]:
    print(k, times[k], sents[k])
print("\nboundaries inside a cue (null times) in lecture 1:")
for k, t in times.items():
    if None in t.values():
        print(k, t, sents[k][:80])

## The LLM API (`scripts/llm/`)

One API for **vLLM** and **llama.cpp**. Both serve the OpenAI chat API, so requests are sent the same way. The parts:

| Name | What it does |
|---|---|
| `LLMConfig` | every setting: engine, model, where the weights are, server flags, sampling, thinking, client |
| `LLM` | starts or attaches to the server, sends requests (`generate`, `generate_many`), caches and logs |
| `Request` | one prompt and its options (schema, check, overrides) |
| `Result` | the answer, thinking, parsed JSON, tokens, time, errors |
| `Server` | builds the server command, starts and stops it (used by `LLM`) |

**Launch or attach.** `LLM(cfg).start()` attaches to a server that already answers at the URL; otherwise (with
`launch=True`) it starts one, waits until it is ready, and `stop()` shuts it down again. A server it attached to is
left running. Set `url` (and `launch=False`) to use a server on another machine, e.g. the EC2.

**Sequential and parallel.** `generate_many(requests, parallel=False)` sends one request at a time.
`parallel=True` sends them all at once, up to `concurrency` (default `slots`) in flight, and the engine batches them:
vLLM by continuous batching, llama.cpp over its slots. Results come back in the order of the requests.

**On this machine** llama.cpp runs inside the `llm-engine` toolbox (`toolbox="llm-engine"`). vLLM runs on the EC2.

**The EC2 (2026-11-12).** The model servers run on the EC2 and the pipeline on this PC, through an SSH port forward
(`ec2.md`; setup: `setup_ec2.sh`). `configs/qwen38_27b_vllm.json` (port 8094) and `configs/sage_a15_vllm.json` (8095)
attach only, waiting up to 30 min for their server; on the EC2, `python3 -m scripts.llm.serve <config>` starts the
server from the same file, so requests and server always agree. `launch` and `attach_wait_s` are not part of the
request hash, so the caches written on the EC2 still answer here.

**Fake model.** `scripts/llm/fake_server.py` answers like vLLM with JSON built from the request's schema (no model).
`tests/fake_run.py` runs 2a and 2b on lectures 1, 4, 16 through the EC2 configs against fake servers on their ports,
in a temporary copy of the pipeline folder, and checks for damage: a wrong model on the port accepted, facts lost or
invented, evidence outside its segment, a rerun not answered from the cache, the rewrite not waiting through the
model swap or not continuing 2a's cached conversation, a fact dropped without a log line, any change under `output/`.


### `LLMConfig` fields

| Group | Field | Default | Meaning |
|---|---|---|---|
| model | `engine` | `"llamacpp"` | `"vllm"` or `"llamacpp"` |
| | `model` | `""` | name requests use (vLLM `--served-model-name`, llama.cpp `--alias`); empty: `model_path` (vLLM) or what the server reports |
| | `model_path` | `""` | weights. vLLM: HF repo id or local folder. llama.cpp: local `.gguf`, `"org/repo:file.gguf"`, or `"org/repo:QUANT"` |
| server | `launch` | `True` | start a server when none answers; `False`: attach only (and the port must serve `model`, else it fails: another model's answers would be cached under this config's hash) |
| | `attach_wait_s` | `0` | attach only: wait this long for a server to answer (while models are swapped on the EC2); `0`: fail at once |
| | `host`, `port` | `127.0.0.1`, `8000` | where the server listens |
| | `url` | `""` | a server elsewhere, e.g. `"http://10.0.0.5:8000"` (overrides host and port; never launched) |
| | `binary` | `""` | `vllm` / `llama-server` executable; empty: found on PATH |
| | `toolbox` | `""` | run the server inside this toolbox container (here: `"llm-engine"`) |
| | `env` | `{}` | extra environment variables for the server |
| | `startup_timeout_s` | `1800` | how long to wait for the server to be ready |
| | `server_log` | `""` | server output file; empty: `output/logs/llm_server_<port>.log` |
| engine | `context_length` | `32768` | vLLM `--max-model-len`; llama.cpp `-c` (one pool shared by the slots with `kv_unified`) |
| | `slots` | `8` | requests run at once: vLLM `--max-num-seqs`, llama.cpp `-np` |
| | `seed` | `0` | server seed; a request's seed is `seed + attempt` |
| | `extra_args` | `[]` | more server flags, passed as they are |
| vLLM | `gpu_memory_utilization` | `0.90` | share of GPU memory vLLM takes |
| | `tensor_parallel_size` | `1` | GPUs per model |
| | `dtype`, `quantization`, `kv_cache_dtype` | `""` | e.g. `"bfloat16"`, `"fp8"`, `"fp8"`; empty: vLLM's choice |
| | `prefix_caching` | `True` | reuse the KV cache of a shared prompt start |
| | `max_num_batched_tokens` | `0` | 0: vLLM's default |
| | `reasoning_parser` | `""` | e.g. `"qwen3"`, `"gemma4"`: thinking returned apart from the answer |
| vLLM + llama.cpp | `think_end` | `""` | words that close the thinking when it reaches `think_budget` |
| llama.cpp | `gpu_layers` | `99` | `-ngl` |
| | `flash_attn` | `True` | `-fa on` |
| | `kv_unified` | `True` | one KV pool for all slots |
| | `jinja` | `True` | the model's own chat template |
| | `reasoning_format` | `"deepseek"` | thinking returned in `reasoning_content` |
| sampling | `temperature`, `top_p`, `top_k`, `min_p` | `0.6`, `0.95`, `20`, `0.0` | per request; a `Request` can override |
| | `presence_penalty`, `repetition_penalty` | `0.0`, `1.0` | |
| | `max_tokens` | `8192` | thinking + answer; 0: no cap |
| thinking | `thinking` | `True` | `enable_thinking` in the chat template (llama.cpp: `--reasoning on/off`) |
| | `think_budget` | `0` | cap on thinking tokens, 0: none. vLLM: per request; llama.cpp: per server (fixed at start) |
| client | `concurrency` | `0` | parallel mode: requests in flight; 0: `slots` |
| | `timeout_s` | `3600` | one request |
| | `retries` | `2` | more attempts when an answer fails its check or is not valid JSON (next seed each time) |
| | `cache_file` | `""` | answers cached by request hash (JSON lines); the same request is answered from the cache |
| | `log_file` | `""` | one JSON line per attempt: tokens, time, finish, errors |

Methods: `cfg.with_(**changes)` (a changed copy), `cfg.save(path)` (JSON), `LLMConfig.load(path)` (`.json` or
`.toml`), `cfg.to_dict()`, `cfg.base_url`.

In [ ]:
from scripts.llm import LLM, LLMConfig, Request, Result, Server

# The config used for run 1: Gemma 4 E4B (Q8 GGUF, already downloaded) on llama.cpp in the llm-engine toolbox.
local = LLMConfig.load("configs/gemma4_e4b_llamacpp.json")
print(json.dumps({k: v for k, v in local.to_dict().items() if v != LLMConfig().to_dict()[k]}, indent=1))
print("\nserver command:\n", " ".join(Server(local).command()))

In [ ]:
# vLLM on the EC2 (v2's large setup: Qwen 3.8 27B FP8). Only builds the command here; nothing is started.
ec2 = LLMConfig(
    engine="vllm",
    model_path="Qwen/Qwen3.8-27B-FP8",
    binary="~/vllm-venv/bin/vllm",
    context_length=65536, slots=128, gpu_memory_utilization=0.92,
    reasoning_parser="qwen3",
    think_end="\n\nI have thought enough; now I give the final answer.\n</think>\n\n",
    think_budget=8192, max_tokens=0,
    temperature=0.6, top_p=0.95, top_k=20,
    extra_args=["--language-model-only"],
)
print(" ".join(Server(ec2).command()))

# Attach to a server that is already running somewhere (no launch):
remote = LLMConfig(engine="vllm", url="http://<ec2-address>:8000", launch=False)

# A changed copy, and saving / loading:
greedy = local.with_(temperature=0.0, think_budget=4096)
# greedy.save("configs/my_config.json"); LLMConfig.load("configs/my_config.json")

### Sending requests

`Request` fields: `prompt`, `key` (your ID, shown in results and logs), `system`, `schema` (JSON schema: the answer is
constrained to it and parsed into `result.parsed`), `check` (a function of the parsed answer that returns a list of
problems; any problem means another attempt), `max_tokens`, `think_budget`, `sampling` (overrides such as
`{"temperature": 0}`), `meta` (returned on the result).

`Result` fields: `key`, `content` (the answer), `reasoning` (the thinking), `parsed`, `ok`, `errors`, `finish`
(`"length"` = cut off by `max_tokens`), `input_tokens`, `output_tokens`, `latency_s`, `attempts`, `cached`, `hash`,
`meta`.

`generate` and `generate_many` take `cache_file=` and `log_file=` to override the config's (stages use their own, e.g.
`output/<course>/stage_1a_segmentation/llm/`). `llm.totals` counts requests, attempts, cache hits and tokens.

The cells below need a running model server (they start the local one, about 9 GB of GPU memory).

In [ ]:
llm = LLM(local).start()          # attaches if it is already running
print(llm.model, "| tokens in 'Hello world':", llm.count_tokens("Hello world"))

r = llm.generate("List the prime numbers between 80 and 120.", max_tokens=4000)
print(r.content, "|", r.output_tokens, "tokens,", f"{r.latency_s:.1f} s, thinking: {len(r.reasoning)} chars")

In [ ]:
# A JSON answer, parsed and checked. A failed check is retried with the next seed (up to cfg.retries times).
primes = Request(
    prompt="List the prime numbers between 80 and 120 as a JSON list of integers.",
    key="primes",
    schema={"type": "array", "items": {"type": "integer"}},
    check=lambda xs: [] if all(80 < x < 120 for x in xs) else ["a number is out of range"],
)
r = llm.generate(primes)
print(r.ok, r.parsed, r.attempts, r.errors)

In [ ]:
# Many requests: one at a time, then all at once.
import time
reqs = [Request(prompt=f"What is {a} times {b}? Answer with the number only.", key=f"{a}x{b}", max_tokens=2000)
        for a, b in [(12, 13), (17, 19), (23, 29), (31, 37), (41, 43), (47, 53)]]
t = time.time(); seq = llm.generate_many(reqs, parallel=False); print(f"sequential: {time.time() - t:.1f} s")
t = time.time(); par = llm.generate_many(reqs, parallel=True); print(f"parallel:   {time.time() - t:.1f} s")
print([(x.key, x.content) for x in par])

In [ ]:
llm.stop()   # stops the server only if this LLM started it

## Stage 1a: segmentation

`scripts/s1a_segmentation.py` splits each lecture into segments with the prompt in `prompts/segmentation.txt`. The
prompt is the task, the course block (outline and syllabus, below), the lecture's title, the definitions and rules,
and the whole lecture, numbered from `[1]`. The model answers with its segments in order (constrained by a JSON schema):

```json
[{"start": 1, "subgoal": "Explain what extended reality means and how VR, AR and MR relate to it."},
 {"start": 38, "subgoal": "Describe how a head-mounted display creates a sense of presence."}]
```

Stage 1a only generates: one answer per lecture, saved as it is. All checks and retries are stage 1b's. (The LLM
API itself re-asks only when an answer is not JSON at all, e.g. cut off at `max_tokens`.)

**Parallel** means every lecture is sent at once and the engine batches them; `parallel=False` sends one at a time.

| File | Content |
|---|---|
| `stage_1a_segmentation/raw/lec_<n>.json` | `answer` (parsed), `content` (as the model gave it; 1b sends it back in a retry), `prompt_sha` (1b checks it rebuilds the same prompt), tokens, time, `reasoning` (the thinking) |
| `stage_1a_segmentation/llm/cache.jsonl`, `calls.jsonl` | answer cache (a re-run is answered from it) and one line per call |

`run(course, llm, lectures=None, parallel=True, max_tokens=None, think_budget=None)`; the last two override the
config for this stage.

Command line: `python scripts/s1a_segmentation.py nptel_vr --llm configs/gemma4_12b_llamacpp.json [--only 1-5] [--sequential]`
(starts the server, or attaches to a running one, and stops what it started).

### The course block (`scripts/course_meta.py`, `prompts/course_meta.txt`)

| Function | Gives |
|---|---|
| `course_meta.load(course)` | `input/<course>/input/course.json` as a dict |
| `course_meta.course_prompt(course)` | the course block: outline and syllabus (topics in order, grouped by theme). A line is left out when the course lacks its field |
| `course_meta.lecture_title(course, n)` | lecture n's title from `titles.tsv` |

A stage's prompt has a `{course}` placeholder for the block. The block is the same for every lecture, so it comes
before anything lecture-specific (the engine's prefix cache reuses it). New course: write its `course.json` by hand
from its syllabus (only `title` is required; the prompts use `outline` and `syllabus`) next to `titles.tsv`.

In [ ]:
from scripts import course_meta, s1a_segmentation
print(course_meta.course_prompt(COURSE))
print()
print(course_meta.lecture_title(COURSE, 17))

In [ ]:
# The full prompt for one lecture (transcript cut short here).
p = s1a_segmentation.build_prompt(s1a_segmentation.load_sentences(COURSE, 36), COURSE, 36)
print(p[:p.index("[4] ")], "...")

In [ ]:
# The config of the current outputs (run 4); with the same config a re-run is answered from llm/cache.jsonl.
seg_cfg = LLMConfig.load("configs/gemma4_12b_llamacpp.json")
with LLM(seg_cfg) as llm:
    seg_summary = s1a_segmentation.run(COURSE, llm)              # every lecture, all at once
    # s1a_segmentation.run(COURSE, llm, lectures=[1, 2], parallel=False)
seg_summary

## Stage 1b: verify segments

`scripts/s1b_verify_segments.py` checks each lecture's answer in code and, when a check fails, sends it back to the
model. Code never merges, splits or re-segments. Once a lecture passes, its segment map is written.

```
answer --> structural checks --fail--> structural retry --> new answer --> structural checks ...
                 | pass
           logical checks ----flagged--> logical retry ----> new answer --> structural checks ...
                 | pass, or logical retries used up
           keep the answer that passes the structural checks with the fewest flagged segments (the later on ties)
```

**Structural checks** (hard: an answer that fails them is never kept). On the answer: a non-empty list; each item has
an integer `start` and a non-empty `subgoal`; the first segment starts at sentence 1; every start is within the lecture;
starts strictly increase. On the segments built from it, each a set of sentences:

| # | Check | Meaning |
|---|---|---|
| 1 | exhaustive | every sentence belongs to some segment |
| 2 | mutually exclusive | no sentence belongs to two segments |
| 3 | contiguous and ordered | each segment is a run of consecutive sentences, following the previous one |
| 4 | non-empty | every segment has at least one sentence |

An answer that passes the answer checks always passes 1–4 (segments are built from ordered starts); 1–4 guard against
bugs.

**Logical checks** (soft, a heuristic): a segment under 2 min or over 15 min looks too small or too big
(`MIN_S`, `MAX_S`). The last segment may be short (the closing remarks may be a segment of their own). The bounds are
not defensible as rules, so the model is told neither them nor the durations: only which segments look too small or
too big, to check them against the instructions, combine small ones into a more meaningful sub-goal or divide big ones
into more manageable sub-goals, and keep a segment that is right. It may push back.

**Budgets.** Up to 2 structural retries for each answer that fails the structural checks in a row; up to 2 logical
retries per lecture (`STRUCTURAL_RETRIES`, `LOGICAL_RETRIES`). A lecture fails (no segment map) only if no answer
passes the structural checks.

**The retry conversation.** The server keeps no state, so 1b sends the whole conversation: the 1a prompt (rebuilt and
checked against 1a's `prompt_sha`), the last answer as the model gave it, and the retry prompt. The model's thinking is
not sent back (5–8k tokens; thinking models are trained to see only earlier answers). Instead every retry answer is a
JSON object `{"reasoning": …, "segments": [...]}`: what it changed, or kept and why. When the last answer was itself a
retry, the next retry prompt repeats the problems it was fixing and its reasoning, so the model builds on its work
instead of starting again.

| Prompt | Placeholders |
|---|---|
| `prompts/segmentation_structural_retry.txt` | `{problems}` |
| `prompts/segmentation_logical_retry.txt` | `{problems}` |

| File | Content |
|---|---|
| `stage_1b_verify_segments/segment_maps/lec_<n>.json` | from the kept answer: `{"1/S0": {"first": "1/0", "last": "1/41", "start": 24.297, "end": 292.435, "duration_s": 268.138, "subgoal": "Explain …"}, …}` |
| `stage_1b_verify_segments/retry/lec_<n>.json` | lectures with retries: every attempt: kind, the retry prompt sent, the answer, its `reasoning`, the model's `thinking`, problems found, segment lengths, tokens |
| `stage_1b_verify_segments/report.json` | per lecture: `ok`, `kept_attempt`, `structural_retries`, `logical_retries`, `flagged_left` (with lengths) |
| `stage_1b_verify_segments/llm/cache.jsonl`, `calls.jsonl` | retry cache and one line per call |

Segment IDs are `<lecture>/S<k>`, `k` from 0. `start` is the first sentence's start and `end` the last sentence's end.

**Times of a cut inside a subtitle cue.** When a segment boundary falls inside a cue (`null` in stage 0a), the segment
map gets an estimated time and says so: `"estimated": ["start"]` (or `["end"]`; the key is there only then). Only
segment boundaries are estimated; sentences keep `null`. The estimate uses the course's median speaking rate (stage
0a's `times/meta.json`) and the cue the boundary falls in (stage 0a's `times/cues/`): forward from the cue's start by
the words spoken before the boundary, backward from the cue's end by the words after it; the mean of the two, kept
inside the cue (`Timing` in the script). The logical checks use the same times. How this was arrived at, with errors
for all six dataset courses: `analysis/preprocessing/report.md`.

`run(course, lectures=None, llm=None)`: `llm` is an `LLM`, or an `LLMConfig` or config file (the server is started
only if a lecture needs a retry); without it 1b only checks.

Command line: `python scripts/s1b_verify_segments.py nptel_vr --llm configs/gemma4_12b_llamacpp.json [--only 1-5]` (without `--llm`: checks only)

In [ ]:
from scripts import s1b_verify_segments
verify_summary = s1b_verify_segments.run(COURSE, llm=seg_cfg)    # retries with the model where a check fails
ids = [f"1/{i}" for i in range(10)]
print(s1b_verify_segments.check([{"start": 2, "subgoal": "x"}, {"start": 2, "subgoal": ""}], ids))
for line in json.loads((OUT / "stage_1b_verify_segments/report.json").read_text())["lec_39"].get("flagged_left", []):
    print(line)

## Stage 1c: segment texts

`scripts/s1c_segment_texts.py` writes one text file per segment from the segment maps:
`stage_1c_segment_texts/lec_<n>/S<k>.txt`, one line per sentence, `[<lecture>/<sentence>] text`.
`write(course, lectures=None)`; command line: `python scripts/s1c_segment_texts.py nptel_vr [--only 1-5]`.

In [ ]:
from scripts import s1c_segment_texts
s1c_segment_texts.write(COURSE)
print((OUT / "stage_1c_segment_texts/lec_1/S0.txt").read_text()[:600])

## Stage 2a: fact extraction

`scripts/s2a_fact_extraction.py` extracts the facts of each segment with `prompts/fact_extraction.txt`, one request
per segment. A fact is one complete point of subject knowledge that the lecturer teaches in the segment: something a
student should know after it and could be asked about in an exam on the subject (definition changed on 2026-11-12
from "a single teachable proposition", see the run notes). Its properties:

| Property | Meaning |
|---|---|
| Significant | what the segment is there to teach: no passing remarks or trivia |
| Whole | one fact per point, not per detail: a definition with what the thing is and is not, a list with all its members and figures, a source with what it is and why it matters; two facts only for points that could be asked about separately |
| Established | the lecturer asserts it: not a hypothesis or a question; a reported view only if it is taught, stated as that view |
| Grounded | it cites every segment sentence needed to state it, and no others |
| Self-contained | readable without the lecture: no "this", "it", "the slide" |
| Terms in full | full words, with the acronym in brackets when the lecturer uses it in the segment: "extended reality (XR)"; never the acronym alone, never one the lecturer does not use |
| On the subject, not on the course | no greetings, logistics, course or lecture outline and objectives, anecdotes, promises of later coverage; a recap only when it adds something |
| Faithful | adds nothing; keeps the lecturer's hedges and who a statement is limited to; names what "this field" means in the lecture; keeps only a corrected version |

Facts are written as subject, relation, predicate. The prompt (two examples: virtual reality, data structures)
gives the course title, the course block (outline and syllabus, as in 1a), the lecture title, the segment
goal (the 1b sub-goal), and the segment's sentences numbered `[1]…[n]`. The answer, constrained by a JSON schema
(evidence numbers within 1..n, at least one):

```json
{"facts": [{"evidence": [1, 2, 3], "text": "Extended reality (XR) is an umbrella term that covers ..."}]}
```

Code maps the numbers back to sentence IDs (local numbers save tokens). Fact IDs are `<segment ID>/F<j>`, `j` from 0.

**Code checks** (`scripts/fact_checks.py`, no LLM), per segment:

| Check | What it does |
|---|---|
| exact repeats | facts with the same content words in the same order become one fact citing all their sentences (its ID keeps the first's place; the repeat's ID is missing) |
| coverage | the share of the segment's sentences cited by some fact: a low share flags under-extraction |
| `pointing` | a pointing word left in the text ("this", "the slide", a leading "It"): 2b sends the fact to the rewrite |
| `acronym` | an acronym in brackets that the segment never uses ("extended reality (ER)"): 2b sends the fact to the rewrite |
| `bare_acronym`, `same_evidence`, `similar` | report only: an acronym without its full term; a fact citing exactly the sentences of another; a fact that reads almost like another (token-sequence similarity 0.8 or more) |

Near-repeats are reported and not collapsed: a list split member by member reads as near-duplicates, and dropping one
would lose its member. The judge decides those (property Whole).

| File | Content |
|---|---|
| `stage_2a_fact_extraction/fact_maps/lec_<n>.json` | `{"1/S0/F0": {"text": "...", "evidence": ["1/9"]}, …}` |
| `stage_2a_fact_extraction/fact_text/lec_<n>.txt` | one line per fact: `[1/S0/F0] text` |
| `stage_2a_fact_extraction/checks/lec_<n>.json` | per segment: sentences, facts, coverage, repeats collapsed, flag counts, flags per fact |
| `stage_2a_fact_extraction/llm/cache.jsonl`, `calls.jsonl` | answer cache (2b rebuilds the 2a conversation from it) and one line per call |

`run(course, llm, lectures=None, parallel=True)`. Command line:
`python scripts/s2a_fact_extraction.py nptel_vr --llm configs/gemma4_12b_llamacpp.json [--only 1-5] [--sequential]`

In [ ]:
from scripts import s2a_fact_extraction
# with LLM(seg_cfg) as llm:
#     s2a_fact_extraction.run(COURSE, llm, lectures=[1])
print((OUT / "stage_2a_fact_extraction/fact_text/lec_1.txt").read_text()[:800])

## Stage 2b: verify facts

`scripts/s2b_verify_facts.py` judges each segment's 2a facts against the eight properties and fixes the ones that
fail. One round per segment:

```
2a facts --> judge (one call per segment) --all pass--> fact map
                   | some fail
             rewrite: 2a conversation rebuilt + failed facts + reasons --> remove, or rewrite (one or more facts)
                   |
             fact map: passed facts in place, rewrites in place of the failed ones, removed ones dropped; renumbered
```

- **Judge** (`prompts/fact_judge.txt`): the course block, lecture title, segment goal, the eight properties; then
  the whole segment numbered `[1]…[n]` and each fact as `[1/S0/F2] (evidence: 3, 4) text`. Answer per fact, in order: `"reason"` (which properties fail and why, or why it passes), then `"pass"`. One overall
  true/false per fact (a per-property verdict can be tested later). The judge sees all the segment's facts, so for
  Whole it names the other facts that are details of the same point; for Grounded it names the sentences to add
  when the fact is right and only its evidence is short.
- **Code checks** (`fact_checks.py`): a fact with a `pointing` or `acronym` flag fails whatever the judge says; the
  check's note is added to the reason (`code_failed` in the judgement record).
- **Rewrite** (`prompts/fact_rewrite.txt`): the 2a conversation rebuilt exactly (2a prompt, 2a's answer as the model
  gave it, found in 2a's cache with `LLM.lookup`), then the failed facts with their evidence and the judge's reason.
  For each: `"remove"`, or `"rewrite"` as a list of one or more facts: only the evidence corrected; facts that are
  details of one point merged (the first rewritten as the whole point, the others removed); a fact holding
  different points split. Rewrites are not judged again (one round, for now); exact repeats among the final facts
  are collapsed.
- A judge answer that misses or repeats a fact, or a rewrite answer that misses a failed fact, is retried by the LLM
  API; a failed fact still without an action is `unresolved` and left out.

| File | Content |
|---|---|
| `stage_2b_verify_facts/fact_maps/lec_<n>.json` | the verified facts, renumbered: `{"1/S0/F0": {"text": …, "evidence": […]}}` |
| `stage_2b_verify_facts/fact_text/lec_<n>.txt` | one line per fact: `[1/S0/F0] text` |
| `stage_2b_verify_facts/judgements/lec_<n>.json` | per segment: verdicts, the rewrite answer, the judge's thinking, and the outcome of every 2a fact (`kept`, `rewritten` with what it became, `removed`, `unresolved`) |
| `stage_2b_verify_facts/checks/lec_<n>.json` | the code checks on the final facts, as in 2a |
| `stage_2b_verify_facts/llm/cache.jsonl`, `calls.jsonl` | answer cache and one line per call |

`run(course, llm, lectures=None)`. Command line:
`python scripts/s2b_verify_facts.py nptel_vr --llm configs/gemma4_12b_llamacpp.json [--only 1-5]`
(2b must use the config 2a was run with: it finds 2a's answers in 2a's cache by request.)

In [ ]:
from scripts import s2b_verify_facts
# with LLM(seg_cfg) as llm:
#     s2b_verify_facts.run(COURSE, llm, lectures=[1])
for r in json.loads((OUT / "stage_2b_verify_facts/judgements/lec_1.json").read_text()):
    for fid, o in r["outcome"].items():
        if o["result"] != "kept":
            print(fid, o)

## Browsing the outputs (`scripts/view.py`)

`Course(course, lectures=None)` loads every stage that has output into dicts, for looking at in this notebook:

| | Gives |
|---|---|
| `c.sentences["1/0"]` | the sentence text |
| `c.times["1/0"]` | `{"start": …, "end": …}` |
| `c.segments["1/S0"]` | the segment, shown as Markdown (header with sentences and time, its sub-goal, then each sentence); `.text` is the plain text, `.sentences` a dict, `.subgoal` |
| `c.segments_of(1)` | the segments of lecture 1 |
| `c.lecture(1)` | a Markdown table of lecture 1's segments with their sub-goals |
| `c.lectures` | lecture numbers |

Times show as m:ss; `?` is a boundary inside a subtitle cue.

In [ ]:
from scripts.view import Course
c = Course(COURSE)
c

In [ ]:
c.sentences["1/0"], c.times["1/0"]

In [ ]:
c.lecture(1)

In [ ]:
c.segments["1/S0"]

## Runs

### Run 1 (2026-11-09): Gemma 4 E4B, local, full course, v2's prompt

Config: `configs/gemma4_e4b_llamacpp.json`: Gemma 4 E4B (unsloth `UD-Q8_K_XL` GGUF) on llama.cpp in the
`llm-engine` toolbox, RTX 5070 12 GB; 6 slots sharing a 131,072-token context; Gemma's own sampling (temperature 1.0,
top_p 0.95, top_k 64); thinking on, capped at 8,192 tokens; `max_tokens` 12,288 (both as in v2). Every lecture sent
at once. Prompts: 5,116 tokens on average, 8,246 at most (lecture 39).

**Results.** 63/63 lectures segmented in 355 s (5.9 min), no failed answers, no retries, nothing cut off by
`max_tokens`, no ignored cut numbers. Thinking: 1,500 output tokens per lecture on average (2,487 at most), well under
the 8,192 cap.

| | Run 1 (Gemma 4 E4B, raw cuts) | v2 large (Qwen 3.8 27B, after merging parts under 2 min) |
|---|--:|--:|
| Segments | 557 (4–16 per lecture, 8.8 on average) | 321 |
| Median duration | 2.3 min | 4.6 min |
| Under 1 min / under 2 min | 77 / 229 | 0 / 0 (merged) |
| Over 10 min / over 15 min | 20 / 2 | 36 / 10 |
| Longest | 18.3 min (38/S2) | 27.9 min (L51/P04) |

Observations:
- Many very short segments are the lecture's closing lines cut off on their own ("So, looking forward to seeing you in
  the next part of the lecture.": 10/S3, 7.6 s), or a short aside.
- 6 cuts fall on a sentence that starts inside a subtitle cue, so 12 segments have a `null` start or end: 1/S4–S5,
  13/S1–S2, 42/S11–S12, 48/S10–S11, 57/S5–S6, 59/S5–S6.


### Run 2 (2026-11-11): Gemma 4 E4B, local, full course, new prompt

Same model and config as run 1; the new segmentation prompt (sub-goals, bridges, stand-alone, opening and closing
remarks); answers checked by stage 1b with retries.

**Results.** 63/63 lectures pass 1b, 622 segments, 594 s (9.9 min). 6 lectures were fixed on retry, all with the same
error: the model left the opening remarks out ("The first segment must start at sentence 1, not at sentence 8"; lectures
14, 18, 22, 45, 48, 54); one retry fixed each. Thinking: 2,378 output tokens per lecture on average (4,261 at most).

| | Run 1 (v2 prompt) | Run 2 (new prompt) |
|---|--:|--:|
| Segments | 557 | 622 |
| Per lecture | 4–16 (mean 8.8) | 5–25 (mean 9.9) |
| Median duration | 2.3 min | 2.3 min |
| Under 1 min / under 2 min | 77 / 229 | 62 / 250 |
| Over 10 min / over 15 min | 20 / 2 | 6 / 1 |
| Longest | 18.3 min | 15.4 min |
| First segment under 2 min | 41 | 29 |
| Last segment under 2 min | 48 | 36 |
| Segments with a `null` time | 12 | 10 |
| Open with a back-reference* | 59 | 55 |
| End on a bridge* | 39 | 34 |

\* rough word-pattern counts ("This…", "So, that…" at a segment start; "Now, let us…", a question at a segment end),
not checks. About two thirds of the cuts agree between the runs (run 2 cuts within 3 sentences of a run 1 cut: 363 of
559).

What the sub-goals show:
- The model treats the opening and closing remarks as sub-goals and labels them as such: a 1-min first segment of
  greeting and recap becomes "Summarize the different types of elements that can be tracked" (17/S0); a 1-min closing
  becomes "Conclude the lecture and identify the topics of subsequent lessons" (5/S6). The rule "never a segment of
  their own" is followed in about half the lectures.
- Items of one list are still cut apart: the stages of a contextual inquiry become 41 s and 11 s segments (36/S3, 36/S4).
- Long worked problems are now split by sub-goal: lecture 51 (v2's 28-min segment) has 10 segments, the longest 7.5 min
  ("Formulate the minimum time path problem using a graph", "Define the edge weights", "Calculate the travel time…",
  "Adapt … into a variation of the TSP", "Detail the algorithmic steps…").

### Run 3 (2026-11-11): Gemma 4 12B, local, full course, same prompt as run 2

Config: `configs/gemma4_12b_llamacpp.json`: Gemma 4 12B QAT (unsloth `UD-Q4_K_XL` GGUF, 6.7 GB), otherwise as run 2,
except the KV cache is q8_0 (`-ctk q8_0 -ctv q8_0`): at f16 it does not fit next to the weights on 12 GB at a
131,072-token context (10.4 GB used with q8_0).

**Results.** 63/63 lectures pass 1b, 355 segments, 2,561 s (43 min). One retry (lecture 22: left out the opening
remarks, fixed). Thinking: 6,818 output tokens per lecture on average; 26 lectures ran past the 8,192-token thinking
budget and were told to answer. Those lectures look no different from the rest (5.6 segments per lecture either way,
19 vs 22 over 10 min), so the budget is not the cause of what follows.

| | Run 2 (4B) | Run 3 (12B) |
|---|--:|--:|
| Segments | 622 | 355 |
| Per lecture | 5–25 (mean 9.9) | 2–17 (mean 5.6) |
| Median duration | 2.3 min | 3.9 min |
| Under 1 min / under 2 min | 62 / 250 | 13 / 62 |
| Over 10 min / over 15 min | 6 / 1 | 41 / 12 |
| Longest | 15.4 min | 27.9 min (51/S1) |
| First segment under 2 min | 29 | 6 |
| Last segment under 2 min | 36 | 18 |
| Segments with a `null` time | 10 | 2 |
| Open with a back-reference* | 55 | 31 |
| End on a bridge* | 34 | 9 |

\* as in run 2. 252 of the 12B's 292 cuts are within 3 sentences of a 4B cut: the 12B keeps a subset of the 4B's cuts.

What the segments show:
- **The 12B follows the stand-alone rule literally, and merging cascades.** Its thinking on lecture 51: "[187] 'So, with
  this basic understanding of the algorithm…' refers to [110-186] … move it back to where that thought begins …
  Therefore, Segments 3 and 4 must be merged", then "[109] 'So, given this background…' … Segment 2 and 3 must be
  merged". Lecture 51 is back to v2's 28-min segment (3 segments). Lectures often become a few chapter-sized
  segments: 8/S1 22 min "immersion and presence", 45/S2 22 min "the four stages of the empirical research process",
  38/S2 18 min "specific vs generic guidelines and Shneiderman's eight golden rules".
- Opening remarks are now mostly folded in (first segment under 2 min: 6), but the closing summary is still often its
  own segment ("Summarize the four stages … and preview future topics", 17/S6; 38/S4; 51/S2).
- Short list-item segments remain in a few lectures (14: four segments under 1 min on input and display types; 36/S4
  28 s; 62).
- The rule "move the cut back to where the thought begins" (EACH SEGMENT MUST STAND ALONE) is what the stronger model
  over-applies: almost every segment opens with a connective ("So, given this…"), so every cut can be argued away.

### Run 4 (2026-11-11): Gemma 4 12B, local, full course, restructured prompt

Same config as run 3. Prompt: task first; course block (outline and syllabus); lecture title; sub-goals defined by the
course goals ("a whole step … the learner's task list"); list rule removed; stand-alone forbids only real dependencies;
closing remarks may be the last segment.

**Results.** 63/63 lectures pass 1b with no retries; 431 segments; 2,331 s (39 min). Thinking: 5,824 output tokens
per lecture on average; 12 lectures ran past the thinking budget (run 3: 26). Prompts: 6,461 input tokens on average.

| | Run 2 (4B) | Run 3 (12B) | Run 4 (12B) |
|---|--:|--:|--:|
| Segments | 622 | 355 | 431 |
| Per lecture | 5–25 (mean 9.9) | 2–17 (mean 5.6) | 2–14 (mean 6.8) |
| Median duration | 2.3 min | 3.9 min | 3.2 min |
| Under 1 min / under 2 min | 62 / 250 | 13 / 62 | 21 / 104 |
| Over 10 min / over 15 min | 6 / 1 | 41 / 12 | 22 / 5 |
| Longest | 15.4 min | 27.9 min | 30.8 min |
| First segment under 2 min | 29 | 6 | 13 |
| Last segment under 2 min | 36 | 18 | 30 |
| Open with a back-reference* | 55 | 31 | 41 |
| End on a bridge* | 34 | 9 | 6 |

\* as in run 2. Cut agreement: 220 of run 4's 368 cuts are within 3 sentences of a run 3 cut; 297 of 368 are near a
run 2 cut.

What the segments show:
- **The connective merges are gone.** Lecture 51 (28 min in run 3) is 8 segments, the longest 7.5 min, each a step
  ("Formulate the navigation problem as a weighted graph", "Calculate travel time … using the natural walking model",
  "Map the navigation problem to a standard TSP", "Describe the step-by-step algorithm"). Lecture 17 reads as the
  tracking stages: IMU sensors, calibration, registration, integration, drift correction, summary.
- **Granularity is inconsistent.** The model chooses the size of a "whole step" differently from lecture to lecture.
  39/S1 is 30.8 min: "selection techniques" as one step, although the lecturer develops three categories at length
  ("Now let us move our attention to the second category of selection techniques …", "The third category …"). 2/S0 is
  26.5 min: the whole lecture but the summary. In lecture 8 the opposite: each of the six qualities of immersion
  (1–2 min each) is its own segment.
- **The closing summary is now almost always its own segment** (55 of 63 lectures end on "Summarize …"), often under a
  minute. Allowed by the prompt.
- **Opening recaps still become their own segment in a few lectures** (34/S0 28 s "Summarize the previous discussion
  …", 37/S0, 43/S0, 44/S0, 59/S0 "Recap …"), against the rule.
- Runs are single samples at temperature 1.0, so part of the difference between runs is sampling noise; how much is
  not measured yet.

### Length-retry test (2026-11-11): 8 lectures of run 4, Gemma 4 12B

Stage 1a now sends back answers with segments under 2 min or over 15 min (the last segment may be short: closing
remarks), up to 2 retries, with `prompts/segmentation_retry.txt`. The bounds are a heuristic, not a rule we can
defend, so the model is told neither the bounds nor the durations: only which segments "look too small" or "look too
big", to check them again against the instructions and keep them if they are right. After the retries the answer with
the fewest such segments is kept (soft check: `Request.advise` in the LLM API; never a failure).

Test: lectures 2, 4, 8, 14, 18, 38, 39, 45 (run 4's answers, from the cache, as first attempts); 505 s.

| Lecture | Run 4 | After retry |
|---|---|---|
| 2 | 2 segments: 26.5-min "application domains and system categories" + summary | 6 (2nd retry): market potential; domains; non-immersive, semi-immersive, fully immersive systems; summary |
| 4 | 17.3-min "three defining characteristics" | kept run 4's answer: retry 1 broke a structural rule, retry 2 split it into two pieces judged too small |
| 8 | 13, seven under 2 min (each quality of immersion its own) | 4: objective and issues; immersion and its six qualities (13.6 min); presence; summary |
| 14 | 9, three under 2 min | 5: non-hand input; processing; visual displays; non-visual output (audio, haptics, motion); summary |
| 18 | 14, seven under 2 min | 7, but some merges serve size, not sub-goals: "3D structure tracking and inertial tracking", "other methods: GPS, motion capture, eye tracking, hybrid" |
| 38 | 16.4-min "Shneiderman's eight golden rules", two under 2 min | 5; the rules split "first four" / "last four" (cut between rule 4 and rule 5, no announced boundary) |
| 39 | 30.8-min "selection techniques" | 9: the lecturer's own categories (hand, pointing, image plane and volumetric selection; hand, indirect, bimanual manipulation) |
| 45 | 16.1-min "framing research questions" | 3 parts: non-testable questions, testable questions, the trade-off |

56 segments → 46; segments outside 2–15 min: 25 → 1 (lecture 4). Most changes follow the lecture's structure
(39, 8, 14, 45); some are made for size (18's grab-bag merges, 38's split of a list in the middle).

### Run 4, verified (2026-11-11): stage 1b with retries on run 4's answers, Gemma 4 12B

1a not re-run: 1b works from run 4's saved answers (prompt unchanged). Flow: structural checks → retry → logical checks
→ retry (up to 2 each), every retry answer with its `reasoning`; the logical retry prompt is the test-2 wording. Then the
maps were rebuilt with estimated times for cuts inside a cue (retries from the cache; one new call, lecture 33, whose
flags changed with the new times). 46 retry calls in all; about 40 min.

| | Run 4 (1a answers) | Run 4 verified (1b) |
|---|--:|--:|
| Segments | 431 | 391 |
| Median duration | 3.2 min | 4.1 min |
| Under 1 min / under 2 min | 21 / 104 | 7 / 30 |
| Over 10 min / over 15 min | 22 / 5 | 16 / 0 |
| Longest | 30.8 min | 14.6 min |
| First segment under 2 min | 13 | 0 |
| Last segment under 2 min | 30 | 29 (the closing summary, allowed) |
| Segments flagged (outside 2–15 min, last may be short) | 80 | 1 |

- 63/63 pass; no structural retries. 37 lectures retried for size: 28 once, 9 twice. Kept: the first answer in 26
  lectures (nothing flagged), the first retry in 29, the second in 8.
- One segment is still flagged, kept by the model: "Describe locomotion modification techniques for mitigation"
  (1 min 43 s).
- Estimated times: 4 segments (2 cuts, lectures 42 and 57), marked `estimated`.
- The `reasoning` field works as intended: the model says what it combined or divided and why, and what it kept
  ("Segment 1 was kept as is because it correctly combines the opening remarks with the first sub-goal …").
- Most changes follow the lecture's structure (39: the lecturer's own selection and manipulation categories; 4: the
  checklist and its three worked examples). Some serve size: 38 splits Shneiderman's eight rules into "first four" and
  "final four"; 18 pairs "acoustic tracking and the categories of vision-based tracking".

### Run-to-run variability and the reports (2026-11-11)

The full pipeline (1a + 1b) was run on 12 lectures with seeds 1–3 (seed 0 = the outputs above). Between seeds,
boundary F1 is 0.75 after 1b (0.70 for 1a's answers), WindowDiff 0.15 (0.23), and the spread of segment counts per
lecture halves with 1b (4.3 → 2.2). The disagreement is about granularity in lectures that enumerate items. Details,
figures and data: `analysis/segmentation/report.md` and `analysis/preprocessing/report.md`.

**Dependency judge, first test (2026-11-11).** A judge prompt (`prompts/segmentation_judge.txt`: can a learner who has
watched the earlier lectures, but not the earlier part of this lecture, start at this segment?) was tried on the 10
most suspicious segment openings and 6 controls. No control was flagged; most suspects resolve their "it"/"this"
within a sentence or two; two borderline cases flipped between runs. Decision: not in the pipeline for now; revisit
after the larger-model (27B) runs. Segmentation is closed: quality is good enough for the next stages. Details:
`analysis/segmentation/report.md`, section 6.

In [ ]:
# Segment statistics for the current outputs (run 4, verified by 1b with retries).
import statistics as st
raws = [json.loads(p.read_text()) for p in sorted((OUT / "stage_1a_segmentation/raw").glob("lec_*.json"),
                                                  key=lambda p: int(p.stem.split("_")[1]))]
report = json.loads((OUT / "stage_1b_verify_segments/report.json").read_text())
segs = list(c.segments.values())
durs = [s.duration_s / 60 for s in segs if s.duration_s is not None]
print(f"lectures pass {sum(e['ok'] for e in report.values())}/{len(report)}, segments {len(segs)}, "
      f"per lecture {len(segs) / max(1, len(c.lectures)):.1f}")
print(f"duration (min): median {st.median(durs):.1f}, min {min(durs):.1f}, max {max(durs):.1f}; "
      f"over 10 min: {sum(d > 10 for d in durs)}, over 15: {sum(d > 15 for d in durs)}, under 1: {sum(d < 1 for d in durs)}")
print("segments with a time inside a cue (null):", [s.id for s in segs if s.duration_s is None])
print("structural retries:", [k for k, e in report.items() if e["structural_retries"]],
      "| logical retries:", [k for k, e in report.items() if e["logical_retries"]],
      "| flagged segments left:", sum(len(e.get("flagged_left", [])) for e in report.values()))
print("1a answers cut off at max_tokens:", [r["lecture"] for r in raws if r["finish"] == "length"])
print("1a output tokens: mean", round(st.mean(r["output_tokens"] for r in raws)), "max", max(r["output_tokens"] for r in raws))
for s in sorted(segs, key=lambda s: -(s.duration_s or 0))[:5]:
    print(f"  longest: {s.id} {s.duration_s / 60:.1f} min, {len(s.sentences)} sentences")

### Facts, lecture 1 (2026-11-11): 2a and 2b tests, Gemma 4 12B

**2a.** Lecture 1's 6 segments, all at once: 48 facts, 220 s, every answer valid JSON on the first try; 3–8k output
tokens per segment (thinking). The copy is in `analysis/fact_extraction/data/history/test1_lec1_gemma12b/`. Seen in
the facts: talk about the lecture or course itself counted as facts ("Extended reality is a term that is often
heard", "The course covers …": all of S0, S5/F3–F8); a definition split into three (S2/F0–F2); predictions and
others' perceptions; one dropped hedge (S4/F11 loses the lecturer's "hopefully"). No acronym slipped through.

**Qwen 3.8 27B (Q5_K_M, `model/`), stopped.** Offloaded to the CPU on this machine (24 of 65 layers on the GPU,
`configs/qwen38_27b_llamacpp.json`): 1.4–1.9 tokens/s per request, about 75–90 min per lecture. Stopped after 47 min;
larger models run on the EC2.

**2b judge test.** Both materials, judge seeds 0–2, one rewrite round on seed 0 (41 calls; median 5.4k output tokens
per judge call). Script `analysis/fact_extraction/scripts/judge_test.py`; report with every fact and verdict:
`analysis/fact_extraction/data/judge_test/report.md`.

| Judge sees | Failed (seed 0 / 1 / 2) | Failed in all seeds | Facts that flip between seeds | Rewrite (seed 0) |
|---|---|--:|--:|---|
| evidence only | 3 / 4 / 7 | 2 | 7 | 3 rewritten, 0 removed |
| whole segment | 7 / 6 / 4 | 3 | 8 | 5 rewritten, 2 removed |

- **Lenient where it matters.** The S0 and S5 course-talk facts pass in every run: their segment goals are about the
  course ("… and the objectives of the course", "Summarize the course objectives …"), so by the on-subject property
  ("about the goal of the segment") they are on-subject. Atomic is checked one way only: the split definition passes.
- **Evidence only misses context.** S1/F6 cites an example that follows a question ("Or do you think of something
  where we can see things in a different way?", 1/70). The evidence-only judge passed it 2 of 3 times; the
  whole-segment judge failed it 3 of 3 and the rewrite removed it (S1/F5 the same).
- **Whole segment raises a few false alarms.** On one seed it failed S3/F1–F3, F6 as not grounded because the
  sentence says "scale", not "reality-virtuality continuum" (grounded read against self-contained); the rewrites
  only added "scale".
- **Works:** the dropped hedge (S4/F11) is caught in all 6 runs; the rewrites are good (S4/F11 split into three
  atomic facts with "hopefully" back).
- **Noisy:** each run fails 3–7 facts, but 7–8 facts flip between seeds.

**Kept:** the whole-segment judge; the evidence-only variant was removed (prompt and option). `stage_2b_verify_facts/`
for lecture 1 is the whole-segment run (seed 0, from the test's cache): 48 facts in, 41 kept, 5 rewritten,
2 removed, 48 out.

**Deferred to the EC2 runs:** a larger model for 2a and 2b, and a judge of a different model from the extractor (to
catch what the extractor's model does not). Proposed and not decided: on-subject defined against the course subject
with talk about the lecture or course excluded (2a and judge); atomic in both directions (a fact must also be a
complete, useful proposition); grounded allowing full names that self-containment requires; a majority over several
judge calls against the noise.